# DEXI Drone LLM Fine-Tuning

Fine-tune a base model for reliable drone tool-calling using QLoRA + Unsloth.

**Supported models:** Qwen2.5-1.5B-Instruct, Llama 3.2-1B-Instruct (set in the model selection cell below).

**Requirements:**
- Google Colab with **T4 GPU** (free tier works)
- Upload **5 files**: `train.jsonl`, `val.jsonl`, `tools.json`, `system_prompt.txt`, `models.json`

**No API keys required.** Training takes ~20-30 minutes.

---

## Setup

First, make sure you're using a **T4 GPU**:  
Runtime → Change runtime type → T4 GPU

### Upload Files

Run this cell twice — once for each directory:

1. **First upload** — select from `training/dataset/`: `train.jsonl`, `val.jsonl`
2. **Second upload** — select from `dexi_llm/config/`: `tools.json`, `system_prompt.txt`, `models.json`

In [ ]:
from google.colab import files

print("Upload #1: select train.jsonl and val.jsonl from training/dataset/")
uploaded = files.upload()
print(f"Got: {list(uploaded.keys())}\n")

print("Upload #2: select tools.json, system_prompt.txt, and models.json from dexi_llm/config/")
uploaded = files.upload()
print(f"Got: {list(uploaded.keys())}")

### Select Model

Change `MODEL_NAME` below to switch between models. Everything else is driven by `models.json`.

In [ ]:
import json

# ===== CHANGE THIS to switch models =====
MODEL_NAME = "qwen2.5-1.5b"  # or "llama3.2-1b"
# =========================================

with open("models.json") as f:
    ALL_MODELS = json.load(f)

MODEL_CONFIG = ALL_MODELS[MODEL_NAME]
print(f"Selected model: {MODEL_NAME}")
print(f"  Base: {MODEL_CONFIG['base_model']}")
print(f"  LoRA r={MODEL_CONFIG['lora']['r']}, alpha={MODEL_CONFIG['lora']['lora_alpha']}")
print(f"  Stop tokens: {MODEL_CONFIG['stop_tokens']}")
print(f"  Output dirs: {MODEL_CONFIG['training']}")

### Install Dependencies

In [ ]:
!pip install unsloth

---

## Load Base Model

Load the selected model in 4-bit quantization. This uses ~2GB VRAM.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_CONFIG["base_model"],
    max_seq_length=2048,
    load_in_4bit=True,
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

## Add LoRA Adapters

QLoRA targeting attention and MLP layers. LoRA rank, alpha, and target modules are driven by `models.json`.

In [ ]:
lora_cfg = MODEL_CONFIG["lora"]

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg["r"],
    lora_alpha=lora_cfg["lora_alpha"],
    target_modules=lora_cfg["target_modules"],
    lora_dropout=lora_cfg["lora_dropout"],
    bias="none",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable / 1e6:.1f}M / {total / 1e6:.0f}M ({100 * trainable / total:.1f}%)")

## Load and Format Dataset

Format each example with the model's chat template and tool definitions in the system prompt.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "val": "val.jsonl",
})

# Load shared config (single source of truth for training + runtime)
with open("tools.json") as f:
    TOOLS = json.load(f)
with open("system_prompt.txt") as f:
    SYSTEM_PROMPT = f.read().strip()

# Build the model-agnostic system block (same logic as runtime build_system_block)
tools_json = "\n".join(
    json.dumps(t, separators=(",", ":")) for t in TOOLS
)
SYSTEM_BLOCK = (
    f"{SYSTEM_PROMPT}\n\n"
    "# Tools\n\n"
    "You may call one or more functions to assist with the user query.\n\n"
    "You are provided with function signatures within <tools></tools> XML tags:\n"
    f"<tools>\n{tools_json}\n</tools>\n\n"
    "For each function call, return a json object with function name and "
    "arguments within <tool_call></tool_call> XML tags:\n"
    "<tool_call>\n"
    '{"name": <function-name>, "arguments": <args-json-object>}\n'
    "</tool_call>"
)

# Chat template tokens from model config
tpl = MODEL_CONFIG["chat_template"]

def format_example(example):
    """Format with model-specific chat template and shared tool definitions.

    Uses the shared system prompt and tool definitions from config/
    so the model sees the exact same context at training and runtime.
    """
    text = tpl["bos"]
    text += tpl["system_prefix"] + SYSTEM_BLOCK + tpl["system_suffix"]
    for msg in example["messages"]:
        role = msg["role"]
        if role == "user":
            text += tpl["user_prefix"] + msg["content"] + tpl["user_suffix"]
        elif role == "assistant":
            text += tpl["assistant_prefix"] + msg["content"] + tpl["assistant_suffix"]

    return {"text": text}

dataset = dataset.map(format_example)

print(f"Train: {len(dataset['train'])} examples")
print(f"Val:   {len(dataset['val'])} examples")
print(f"\n--- Sample (first 500 chars) ---")
print(dataset["train"][0]["text"][:500])

## Train

3 epochs with effective batch size of 16 (4 × 4 gradient accumulation). Should take ~20-30 minutes on T4.

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments

# Only compute loss on assistant responses, not the repeated system prompt.
# Without this, the model spends all its learning capacity memorizing the
# system prompt (~1540 tokens) and barely trains on the tool-call format (~100 tokens).
response_template = tpl["assistant_prefix"]
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    data_collator=collator,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        eval_strategy="epoch",
        output_dir=MODEL_CONFIG["training"]["lora_dir"],
        seed=42,
        report_to="none",
    ),
)

print("Starting training...")
results = trainer.train()
print(f"\nTraining complete!")
print(f"Train loss: {results.training_loss:.4f}")

## Test the Model

Quick sanity check before exporting — try a few prompts to verify tool-call format.

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "make the led green",
    "fly forward 2 meters",
    "what's my battery level?",
    "do a flip",
    "play some music",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_BLOCK},
        {"role": "user", "content": prompt},
    ]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    has_tool_call = "<tool_call>" in response
    status = "tool_call" if has_tool_call else "text only"
    print(f"\n{'='*60}")
    print(f"User: {prompt}")
    print(f"Status: {status}")
    print(f"Response: {response[:300]}")

## Export to GGUF

Merge LoRA weights and export as Q4_K_M GGUF (~1GB) for llama.cpp deployment.

In [ ]:
merged_dir = MODEL_CONFIG["training"]["merged_dir"]
gguf_dir = MODEL_CONFIG["training"]["gguf_dir"]

print(f"Merging LoRA weights to {merged_dir}...")
model.save_pretrained_merged(merged_dir, tokenizer)

print(f"Exporting to GGUF (Q4_K_M) in {gguf_dir}...")
model.save_pretrained_gguf(
    gguf_dir,
    tokenizer,
    quantization_method="q4_k_m",
)

# Find the exported GGUF file
import os, glob
gguf_files = glob.glob(f"{gguf_dir}*/**/*.gguf", recursive=True)
if gguf_files:
    gguf_path = gguf_files[0]
    size_mb = os.path.getsize(gguf_path) / (1024 * 1024)
    print(f"\nGGUF exported: {gguf_path} ({size_mb:.0f} MB)")
else:
    print("\nWarning: GGUF file not found. Check output above for the path.")

## Download

Download the GGUF file to your local machine, then deploy:

**Simulation (Docker):**
```bash
cp unsloth.Q4_K_M.gguf dexi_ws/src/dexi_llm/models/<gguf_name>
```

**Raspberry Pi:**
```bash
scp unsloth.Q4_K_M.gguf dexi@192.168.68.59:~/dexi_ws/src/dexi_llm/models/<gguf_name>
```

Then launch (use `model_name:=llama3.2-1b` if you trained Llama):
```bash
ros2 launch dexi_llm llm_node.launch.py backend:=llama_cpp \
    model_path:=<path>/<gguf_name> \
    model_name:=<model_name>
```

In [ ]:
from google.colab import files
import glob
gguf_dir = MODEL_CONFIG["training"]["gguf_dir"]
gguf_files = glob.glob(f"{gguf_dir}*/**/*.gguf", recursive=True)
print(f"Downloading: {gguf_files[0]}")
print(f"Rename to: {MODEL_CONFIG['gguf_name']}")
files.download(gguf_files[0])